**Geochemistry Biplot App for Bruker Results.csv Files**

Voila App Version
N. Tripcevich 2026, CC BY-SA 4.0  
[More Information Online](https://github.com/arf-berkeley/bruker-xrf-ppm-plot)


In [ ]:
# ============================================================
# Cell 1 - Imports and Configuration
# ============================================================
import sys
import os
import re
import functools
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
import ipywidgets as widgets
from io import StringIO
import subprocess
import importlib

def _ensure(package, import_name=None):
    name = import_name or package
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', package, '-q'],
            capture_output=True
        )

_ensure('plotly')
_ensure('ipywidgets')

ALL_APPS    = 'All applications'
ALL_BATCHES = 'All'
LOD_STRINGS = {'< LOD', '1.#SNB', 'None', '', 'none', 'NONE'}

META_COLS = [
    'File #', 'DateTime', 'Operator', 'Name', 'ID',
    'Field1', 'Field2', 'Application', 'Method',
    'ElapsedTime', 'Multiplier', 'Cal Check',
    'Alloy 1', 'Match Qual 1', 'Alloy 2', 'Match Qual 2',
    'Alloy 3', 'Match Qual 3', '_batch'
]

PPM_SCALE         = 10000
DEFAULT_BIPLOT_X  = 'Sr'
DEFAULT_BIPLOT_Y  = 'Rb'
DEFAULT_TERNARY_A = 'Rb'
DEFAULT_TERNARY_B = 'Sr'
DEFAULT_TERNARY_C = 'Zr'
ELEMENT_PATTERN   = re.compile(r'^[A-Z][a-z]?$')

class StudyState:
    raw          = None
    filtered     = None
    study        = None
    element_cols = []
    plot_df      = None
    hover_cols   = []
    name_order   = []

state = StudyState()

display(widgets.HTML(
    '<span style="color:green;font-size:13px">'
    '&#10003; Cell 1 ready — imports and configuration loaded.'
    '</span>'
))

**Select the Results.csv table from your Bruker analysis**

Browse to a copy of the __Results.csv__ file typically found in Bruker/Data/Results.csv

Importing all of the Weight Percent data from the most recent Method used.

In [ ]:
# ============================================================
# Cell 2 - Upload and Filter
# Supports: VS Code (local) | Binder | Voila
# ============================================================

# ── Pre-display output container at cell level for Voila ──────────────────
main_out  = widgets.Output()
parse_out = widgets.Output()
display(widgets.HTML('<h3>Step 1 - Upload and Filter</h3>'))
display(main_out)

# ── Environment detection ─────────────────────────────────────────────────
@functools.lru_cache(maxsize=None)
def is_local():
    hosted = [
        'BINDER_LAUNCH_HOST', 'JUPYTERHUB_USER',
        'JUPYTERHUB_SERVICE_PREFIX', 'COLAB_BACKEND_VERSION',
        'VOILA_APP_PORT', 'SERVER_SOFTWARE',
    ]
    if any(os.environ.get(v) for v in hosted):
        return False
    if 'voila' in sys.modules:
        return False
    try:
        import tkinter as tk
        root = tk.Tk()
        root.withdraw()
        root.destroy()
        return True
    except Exception:
        return False

# ── Parser ────────────────────────────────────────────────────────────────
def parse_bruker_results(content_str, verbose=True):
    def _log(*args):
        if verbose:
            with parse_out:
                print(*args)

    content_str = content_str.replace('\r\n', '\n').replace('\r', '\n')
    lines       = content_str.split('\n')

    segments        = []
    current_headers = None
    current_rows    = []

    for line in lines:
        line = line.strip()
        if not line:
            if current_headers and current_rows:
                segments.append((current_headers, current_rows))
                current_rows = []
            continue

        cols = [c.strip() for c in line.split(',')]

        if cols[0] == 'File #':
            if current_headers and current_rows:
                segments.append((current_headers, current_rows))
                current_rows = []
            current_headers = cols
            continue

        if current_headers:
            current_rows.append(cols)

    if current_headers and current_rows:
        segments.append((current_headers, current_rows))

    if not segments:
        raise ValueError(
            'No segments found. '
            'Expected CSV with "File #" header rows.'
        )

    frames = []
    for seg_idx, (headers, rows) in enumerate(segments, start=1):
        records = []
        for row in rows:
            if len(row) < len(headers):
                row = row + [''] * (len(headers) - len(row))
            elif len(row) > len(headers):
                row = row[:len(headers)]

            record = {}
            for col, val in zip(headers, row):
                val = val.strip()
                if val in LOD_STRINGS or val == '':
                    record[col] = np.nan
                else:
                    try:
                        record[col] = float(val)
                    except ValueError:
                        record[col] = val
            records.append(record)

        if records:
            df_seg           = pd.DataFrame(records)
            df_seg['_batch'] = seg_idx
            frames.append(df_seg)

    if not frames:
        raise ValueError('No data rows found after parsing.')

    df = pd.concat(frames, ignore_index=True, join='outer')

    if 'Application' in df.columns:
        app_series   = df['Application'].fillna('').astype(str)
        df['_batch'] = (app_series != app_series.shift()).cumsum()

    apps = (
        df['Application'].dropna().unique().tolist()
        if 'Application' in df.columns else []
    )
    _log(
        f'Loaded {len(df)} rows | '
        f'{df["_batch"].nunique()} batches | '
        f'Applications: {apps}'
    )
    return df

# ── Upload decoder ────────────────────────────────────────────────────────
def _decode_upload(upload_widget):
    val = upload_widget.value
    if isinstance(val, dict):
        if not val:
            raise ValueError('No file uploaded.')
        content_data = next(iter(val.values()))['content']
    elif isinstance(val, (list, tuple)):
        if not val:
            raise ValueError('No file uploaded.')
        content_data = val[0]['content']
    else:
        raise ValueError(f'Unrecognised upload type: {type(val)}')
    if isinstance(content_data, memoryview):
        raw = bytes(content_data)
    else:
        raw = bytes(content_data)
    return raw.decode('utf-8-sig', errors='replace')

# ── Filter UI ─────────────────────────────────────────────────────────────
def build_filter_ui(host_out):
    if state.raw is None:
        with host_out:
            print('No data loaded.')
        return

    state.filtered = state.raw.copy()

    def get_unique(col):
        if col not in state.raw.columns:
            return []
        return sorted(
            state.raw[col].dropna().astype(str)
                 .str.strip().replace('', np.nan)
                 .dropna().unique().tolist()
        )

    def batches_for(application):
        df = state.raw.copy()
        if application != ALL_APPS:
            df = df[df['Application'].astype(str).str.strip() == application.strip()]
        return [ALL_BATCHES] + [
            str(b) for b in sorted(df['_batch'].dropna().unique())
        ]

    all_apps    = [ALL_APPS] + get_unique('Application')
    default_app = all_apps[1] if len(all_apps) > 1 else ALL_APPS

    app_dd = widgets.Dropdown(
        options=all_apps, value=default_app,
        description='Application:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='360px')
    )
    batch_dd = widgets.Dropdown(
        options=batches_for(default_app), value=ALL_BATCHES,
        description='Batch:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='180px')
    )
    apply_btn = widgets.Button(
        description='Apply Filter',
        button_style='primary',
        icon='filter',
        layout=widgets.Layout(width='150px')
    )
    summary_out = widgets.Output()
    filter_out  = widgets.Output()

    def refresh_summary(application):
        with summary_out:
            summary_out.clear_output(wait=True)
            df = state.raw.copy()
            if application != ALL_APPS:
                df = df[
                    df['Application'].astype(str).str.strip()
                    == application.strip()
                ]
            batches = sorted(df['_batch'].unique().tolist())
            dates   = pd.to_datetime(
                df['DateTime'], errors='coerce'
            ).dropna()
            d_min = dates.min().strftime('%m-%d-%Y') if not dates.empty else '?'
            d_max = dates.max().strftime('%m-%d-%Y') if not dates.empty else '?'
            print(f'  "{application}" -> {len(df)} rows')
            print(f'  Batches   : {batches}')
            print(f'  Dates     : {d_min} - {d_max}')

    def on_app_change(change):
        batch_dd.options = batches_for(change['new'])
        batch_dd.value   = ALL_BATCHES
        refresh_summary(change['new'])

    def on_apply(b):
        filter_out.clear_output(wait=True)
        with filter_out:
            df        = state.raw.copy()
            sel_app   = app_dd.value
            sel_batch = batch_dd.value

            if sel_app != ALL_APPS:
                df = df[
                    df['Application'].astype(str).str.strip()
                    == sel_app.strip()
                ]
            if sel_batch != ALL_BATCHES:
                try:
                    df = df[df['_batch'] == int(sel_batch)]
                except ValueError:
                    print(f'Invalid batch: {sel_batch}')
                    return

            df             = df.reset_index(drop=True)
            state.filtered = df

            if df.empty:
                print(f'No rows for app="{sel_app}" batch="{sel_batch}"')
                return

            dates = pd.to_datetime(
                df['DateTime'], errors='coerce'
            ).dropna()
            d_min = dates.min().strftime('%m-%d-%Y %H:%M') if not dates.empty else '?'
            d_max = dates.max().strftime('%m-%d-%Y %H:%M') if not dates.empty else '?'

            try:
                file_nums = df['File #'].dropna().astype(int)
                f_min, f_max = int(file_nums.min()), int(file_nums.max())
            except Exception:
                f_min = f_max = '?'

            print(f'✓ {len(df)} rows kept')
            print(f'  Application : {sel_app}')
            print(f'  Batch       : {sel_batch}')
            print(f'  File #      : {f_min} - {f_max}')
            print(f'  Dates       : {d_min} - {d_max}')
            print('\nProceed to Step 2 Clean Data.')

    app_dd.observe(on_app_change, names='value')
    apply_btn.on_click(on_apply)

    with host_out:
        host_out.clear_output(wait=True)
        display(parse_out)
        display(widgets.HTML(
            '<b>Filter by Application and Batch</b><br>'
            '<span style="color:grey;font-size:12px">'
            'Select an Application. Batch updates automatically. '
            'Then click Apply Filter.</span>'
        ))
        display(widgets.VBox([
            widgets.HBox([app_dd, batch_dd, apply_btn]),
            summary_out,
            filter_out
        ]))
    refresh_summary(app_dd.value)

# ── Load local ────────────────────────────────────────────────────────────
def load_local():
    import tkinter as tk
    from tkinter import filedialog
    if state.raw is not None:
        with main_out:
            print(f'Already loaded: {state.raw.shape[0]} rows')
        build_filter_ui(main_out)
        return
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    path = filedialog.askopenfilename(
        title='Select Results.csv',
        filetypes=[('CSV files', '*.csv'), ('All files', '*.*')]
    )
    root.destroy()
    with main_out:
        if path:
            try:
                with open(path, 'r', encoding='utf-8-sig', errors='replace') as f:
                    state.raw = parse_bruker_results(f.read())
                print(f'Loaded: {path}')
                build_filter_ui(main_out)
            except Exception as e:
                print(f'Failed to load: {e}')
        else:
            print('No file selected. Re-run cell to try again.')

# ── Load hosted ───────────────────────────────────────────────────────────
def load_hosted():
    if state.raw is not None:
        with main_out:
            print(f'Already loaded: {state.raw.shape[0]} rows')
        build_filter_ui(main_out)
        return

    upload_widget = widgets.FileUpload(accept='.csv', multiple=False)
    load_btn      = widgets.Button(
        description='Load Data',
        button_style='success',
        icon='check',
        disabled=True
    )
    status_lbl  = widgets.Label('Upload a Bruker XRF Results.csv file')
    session_lbl = widgets.HTML(
        '<span style="color:grey;font-size:12px">'
        'Sessions are temporary - re-upload each session.</span>'
    )
    filter_area = widgets.Output()

    def _on_upload_change(change):
        if upload_widget.value:
            load_btn.disabled = False
            try:
                val  = upload_widget.value
                size = (
                    next(iter(val.values())).get('size', 0)
                    if isinstance(val, dict)
                    else len(val[0].get('content', b''))
                )
                size_mb          = size / (1024 * 1024)
                status_lbl.value = (
                    f'Large file ({size_mb:.1f} MB) - may be slow. Click Load Data.'
                    if size_mb > 50
                    else f'File ready ({size_mb:.1f} MB) - click Load Data'
                )
            except Exception:
                status_lbl.value = 'File ready - click Load Data'
        else:
            load_btn.disabled = True

    def _on_load(b):
        try:
            content              = _decode_upload(upload_widget)
            state.raw            = parse_bruker_results(content)
            status_lbl.value     = f'Loaded {state.raw.shape[0]} rows'
            load_btn.disabled    = True
            load_btn.description = 'Loaded'
            with filter_area:
                filter_area.clear_output(wait=True)
                build_filter_ui(filter_area)
        except Exception as e:
            status_lbl.value = f'Error: {e}'

    upload_widget.observe(_on_upload_change, names='value')
    load_btn.on_click(_on_load)

    with main_out:
        display(widgets.VBox([
            widgets.Label('Upload Results.csv:'),
            upload_widget,
            load_btn,
            status_lbl,
            session_lbl,
            filter_area
        ]))

# ── Entry point ───────────────────────────────────────────────────────────
if is_local():
    load_local()
else:
    load_hosted()

Cleaning data includes removing the following: elemental error columns, Alloy, Match Qual columns, Multiplier, Cal Check, Operator, Field 1&2. This script also replaces Below Detection Limits LOD with 0.

You may now select rows of data from recent analyses by filtering with either File # or Date.

In [ ]:
# ============================================================
# Cell 3 - Clean Data
# Supports: VS Code (local) | Binder | Voila
# ============================================================

cell3_out = widgets.Output()
clean_btn = widgets.Button(
    description='Clean Data',
    button_style='primary',
    icon='cog',
    layout=widgets.Layout(width='160px')
)

display(widgets.HTML('<h3>Step 2 - Clean Data</h3>'))
display(clean_btn)
display(cell3_out)

def on_clean(btn):
    cell3_out.clear_output(wait=True)
    with cell3_out:
        if state.filtered is None or len(state.filtered) == 0:
            print('Please upload and filter data in Step 1 first.')
            return

        _source = state.filtered.copy()
        print(f'Using filtered data : {len(_source)} rows')
        print(f'Application(s)      : '
              f'{_source["Application"].dropna().unique().tolist()}')
        print(f'Batch(es)           : '
              f'{sorted(_source["_batch"].unique().tolist())}')

        # ── Drop non-element columns ───────────────────────────────────────
        drop_cols = [
            c for c in _source.columns
            if c in META_COLS or 'Err' in c
        ]
        keep_cols = [c for c in _source.columns if c not in drop_cols]
        study     = _source[keep_cols].copy()

        # ── Identify column types ──────────────────────────────────────────
        string_cols = [
            c for c in ['Name', 'Application', 'Method']
            if c in study.columns
        ]
        meta_numeric = ['File #', 'ElapsedTime']

        element_cols = [
            c for c in study.columns
            if ELEMENT_PATTERN.match(c)
            and c not in string_cols
            and c not in meta_numeric
            and c != 'DateTime'
        ]
        numeric_cols = [
            c for c in study.columns
            if c not in string_cols
            and c != 'DateTime'
        ]

        # ── Type casting ───────────────────────────────────────────────────
        study[string_cols]  = study[string_cols].astype('string')
        study[numeric_cols] = study[numeric_cols].apply(
            pd.to_numeric, errors='coerce'
        )
        if 'DateTime' in study.columns:
            study['DateTime'] = pd.to_datetime(
                study['DateTime'], errors='coerce'
            )

        # ── Scale to ppm and fill LOD with 0 ──────────────────────────────
        study[element_cols] = (study[element_cols] * PPM_SCALE).round(1)
        study[element_cols] = study[element_cols].fillna(0)

        # ── Drop empty rows and columns ────────────────────────────────────
        study.dropna(axis=1, how='all', inplace=True)
        study.dropna(how='all', inplace=True)
        study = study.reset_index(drop=True)

        # ── Re-identify element cols after dropna ──────────────────────────
        element_cols = [c for c in element_cols if c in study.columns]

        # ── Build plot-ready dataframe ─────────────────────────────────────
        plot_df = study.copy()
        for col in plot_df.columns:
            if pd.api.types.is_string_dtype(plot_df[col]) or \
               pd.api.types.is_object_dtype(plot_df[col]):
                plot_df[col] = (
                    plot_df[col]
                    .astype(object)
                    .fillna('(no name)')
                    .astype(str)
                    .replace({
                        'nan'  : '(no name)',
                        'None' : '(no name)',
                        '<NA>' : '(no name)',
                        ''     : '(no name)'
                    })
                )

        if 'Name' not in plot_df.columns:
            plot_df['Name'] = '(no name)'
        else:
            plot_df['Name'] = (
                plot_df['Name']
                .astype(str)
                .replace({
                    'nan'  : '(no name)',
                    'None' : '(no name)',
                    '<NA>' : '(no name)',
                    ''     : '(no name)'
                })
            )

        for col in element_cols:
            plot_df[col] = pd.to_numeric(
                plot_df[col], errors='coerce'
            ).fillna(0)

        if 'DateTime' in plot_df.columns:
            plot_df['DateTime'] = (
                pd.to_datetime(plot_df['DateTime'], errors='coerce')
                .dt.strftime('%m-%d-%Y %H:%M')
                .fillna('unknown')
            )

        if 'File #' in plot_df.columns:
            plot_df['File #'] = (
                pd.to_numeric(plot_df['File #'], errors='coerce')
                .fillna(0)
                .astype(int)
                .astype(str)
            )

        hover_cols = [
            c for c in ['File #', 'DateTime', 'Application', 'Method']
            if c in plot_df.columns
        ]
        name_order = sorted(plot_df['Name'].unique().tolist())

        # ── Store on state ─────────────────────────────────────────────────
        state.study        = study
        state.element_cols = element_cols
        state.plot_df      = plot_df
        state.hover_cols   = hover_cols
        state.name_order   = name_order

        # ── Summary ────────────────────────────────────────────────────────
        print(f'\nRows after cleaning : {len(study)}')
        print(f'Element columns     : {element_cols}')
        print(f'Name values         : {name_order}')
        print('\nProceed to Step 3 Biplot or Step 4 Ternary.')

clean_btn.on_click(on_clean)

In [ ]:
# ============================================================
# Cell 4 - Biplot
# Supports: VS Code (local) | Binder | Voila
# ============================================================

cell4_out   = widgets.Output()
biplot_btn  = widgets.Button(
    description='Draw Biplot',
    button_style='success',
    icon='bar-chart',
    layout=widgets.Layout(width='160px')
)
biplot_plot = widgets.Output()

display(widgets.HTML('<h3>Step 3 - Biplot</h3>'))
display(biplot_btn)
display(cell4_out)
display(biplot_plot)

def on_biplot(btn):
    cell4_out.clear_output(wait=True)
    biplot_plot.clear_output(wait=True)
    with cell4_out:
        if state.plot_df is None:
            print('Please run Step 2 Clean Data first.')
            return

        element_cols = state.element_cols
        if not element_cols:
            print('No element columns found. Check Step 2.')
            return

        default_x = (
            DEFAULT_BIPLOT_X if DEFAULT_BIPLOT_X in element_cols
            else element_cols[0]
        )
        default_y = (
            DEFAULT_BIPLOT_Y if DEFAULT_BIPLOT_Y in element_cols
            else element_cols[1] if len(element_cols) > 1
            else element_cols[0]
        )

        x_dd = widgets.Dropdown(
            options=element_cols,
            value=default_x,
            description='X Axis:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='200px')
        )
        y_dd = widgets.Dropdown(
            options=element_cols,
            value=default_y,
            description='Y Axis:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='200px')
        )

        def update_biplot(change=None):
            with biplot_plot:
                biplot_plot.clear_output(wait=True)
                x          = x_dd.value
                y          = y_dd.value
                plot_df    = state.plot_df.copy()
                hover_cols = state.hover_cols
                name_order = state.name_order

                plot_df = plot_df[
                    (plot_df[x] > 0) & (plot_df[y] > 0)
                ]
                if plot_df.empty:
                    print(f'No rows with valid data for {x} and {y}.')
                    return
                try:
                    fig = px.scatter(
                        plot_df, x=x, y=y,
                        color='Name',
                        category_orders={'Name': name_order},
                        hover_data=[
                            c for c in hover_cols
                            if c != x and c != y
                        ],
                        title=f'{y} vs {x} Biplot',
                        labels={
                            x      : f'{x} (ppm)',
                            y      : f'{y} (ppm)',
                            'Name' : 'Sample Name'
                        }
                    )
                    fig.update_traces(
                        marker=dict(size=8, opacity=0.85)
                    )
                    fig.update_layout(
                        height=600,
                        hovermode='closest',
                        legend=dict(
                            title=dict(
                                text='Sample Name',
                                font=dict(size=13)
                            ),
                            itemsizing='constant',
                            bordercolor='lightgrey',
                            borderwidth=1,
                            bgcolor='rgba(255,255,255,0.85)',
                            x=1.02, xanchor='left',
                            y=1,    yanchor='top'
                        ),
                        margin=dict(r=180)
                    )
                    display(fig)
                except Exception as e:
                    print(f'Plot error: {e}')

        x_dd.observe(update_biplot, names='value')
        y_dd.observe(update_biplot, names='value')

        display(widgets.HBox([x_dd, y_dd]))
        update_biplot()

biplot_btn.on_click(on_biplot)

In [ ]:
# ============================================================
# Cell 5 - Ternary Plot
# Supports: VS Code (local) | Binder | Voila
# ============================================================

cell5_out    = widgets.Output()
ternary_btn  = widgets.Button(
    description='Draw Ternary',
    button_style='success',
    icon='signal',
    layout=widgets.Layout(width='160px')
)
ternary_plot = widgets.Output()

display(widgets.HTML('<h3>Step 4 - Ternary Plot</h3>'))
display(ternary_btn)
display(cell5_out)
display(ternary_plot)

def on_ternary(btn):
    cell5_out.clear_output(wait=True)
    ternary_plot.clear_output(wait=True)
    with cell5_out:
        if state.plot_df is None:
            print('Please run Step 2 Clean Data first.')
            return

        element_cols = state.element_cols
        if len(element_cols) < 3:
            print(f'Need at least 3 elements. Found: {element_cols}')
            return

        default_a = (
            DEFAULT_TERNARY_A if DEFAULT_TERNARY_A in element_cols
            else element_cols[0]
        )
        default_b = (
            DEFAULT_TERNARY_B if DEFAULT_TERNARY_B in element_cols
            else element_cols[1]
        )
        default_c = (
            DEFAULT_TERNARY_C if DEFAULT_TERNARY_C in element_cols
            else element_cols[2]
        )

        a_dd = widgets.Dropdown(
            options=element_cols,
            value=default_a,
            description='A (top):',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='220px')
        )
        b_dd = widgets.Dropdown(
            options=element_cols,
            value=default_b,
            description='B (bottom left):',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='220px')
        )
        c_dd = widgets.Dropdown(
            options=element_cols,
            value=default_c,
            description='C (bottom right):',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='220px')
        )

        def update_ternary(change=None):
            with ternary_plot:
                ternary_plot.clear_output(wait=True)
                a = a_dd.value
                b = b_dd.value
                c = c_dd.value

                if len({a, b, c}) < 3:
                    print('Please select three different elements.')
                    return

                plot_df    = state.plot_df.copy()
                hover_cols = state.hover_cols
                name_order = state.name_order

                plot_df = plot_df[
                    (plot_df[a] > 0) &
                    (plot_df[b] > 0) &
                    (plot_df[c] > 0)
                ]
                if plot_df.empty:
                    print(f'No rows with valid data for {a}, {b}, {c}.')
                    return
                try:
                    fig = px.scatter_ternary(
                        plot_df, a=a, b=b, c=c,
                        color='Name',
                        category_orders={'Name': name_order},
                        hover_data=[
                            col for col in hover_cols
                            if col not in (a, b, c)
                        ],
                        title=f'Ternary: {a} / {b} / {c}',
                        labels={
                            'Name' : 'Sample Name',
                            a      : f'{a} (ppm)',
                            b      : f'{b} (ppm)',
                            c      : f'{c} (ppm)'
                        }
                    )
                    fig.update_traces(
                        marker=dict(size=8, opacity=0.85)
                    )
                    fig.update_layout(
                        height=650,
                        legend=dict(
                            title=dict(
                                text='Sample Name',
                                font=dict(size=13)
                            ),
                            itemsizing='constant',
                            bordercolor='lightgrey',
                            borderwidth=1,
                            bgcolor='rgba(255,255,255,0.85)',
                            x=1.02, xanchor='left',
                            y=1,    yanchor='top'
                        ),
                        margin=dict(r=180)
                    )
                    display(fig)
                except Exception as e:
                    print(f'Plot error: {e}')

        a_dd.observe(update_ternary, names='value')
        b_dd.observe(update_ternary, names='value')
        c_dd.observe(update_ternary, names='value')

        display(widgets.HBox([a_dd, b_dd, c_dd]))
        update_ternary()

ternary_btn.on_click(on_ternary)

In [ ]:
# ============================================================
# Cell 6 - Export CSV
# Supports: VS Code (local) | Binder | Voila
# ============================================================
import base64
from IPython.display import HTML as IHTML

cell6_out   = widgets.Output()
export_btn  = widgets.Button(
    description='Export CSV',
    button_style='warning',
    icon='download',
    layout=widgets.Layout(width='160px')
)
export_link = widgets.Output()

display(widgets.HTML('<h3>Step 5 - Export CSV</h3>'))
display(export_btn)
display(cell6_out)
display(export_link)

def on_export(btn):
    cell6_out.clear_output(wait=True)
    export_link.clear_output(wait=True)
    with cell6_out:
        if state.study is None:
            print('Please run Step 2 Clean Data first.')
            return

        df = state.study

        try:
            dates    = pd.to_datetime(
                df['DateTime'], errors='coerce'
            ).dropna()
            date_str = (
                dates.max().strftime('%Y%m%d')
                if not dates.empty else 'export'
            )
        except Exception:
            date_str = 'export'

        try:
            app_str = (
                df['Application']
                .dropna()
                .unique()[0]
                .replace(' ', '_')
                if 'Application' in df.columns
                else 'Bruker'
            )
        except Exception:
            app_str = 'Bruker'

        filename = f'Bruker_{app_str}_{date_str}.csv'
        csv_str  = df.to_csv(index=False)
        b64      = base64.b64encode(csv_str.encode()).decode()

        with export_link:
            display(IHTML(
                f'<a download="{filename}" '
                f'href="data:text/csv;base64,{b64}" '
                f'style="font-size:14px;font-weight:bold;">'
                f'Click here to download {filename}</a>'
            ))

        print(f'Ready     : {filename}')
        print(f'Rows      : {len(df)}')
        print(f'Columns   : {df.columns.tolist()}')

export_btn.on_click(on_export)